## Predicting Osteoporosis Risk

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Statistical analysis
from scipy.stats import chi2_contingency

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

In [3]:
df = pd.read_csv("osteoporosis.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (1958, 16)


,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
0,104866,69,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,Yes,Moderate,Rheumatoid Arthritis,Corticosteroids,Yes,1
1,101999,32,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,No,NaN,NaN,NaN,Yes,1
2,106567,89,Female,Postmenopausal,No,Caucasian,Normal,Adequate,Sufficient,Active,No,Moderate,Hyperthyroidism,Corticosteroids,No,1
3,102316,78,Female,Normal,No,Caucasian,Underweight,Adequate,Insufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,Corticosteroids,No,1
4,101944,38,Male,Postmenopausal,Yes,African American,Normal,Low,Sufficient,Active,Yes,NaN,Rheumatoid Arthritis,NaN,Yes,1


In [4]:
# Basic data understanding

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())

Rows: 1958
Columns: 16

Column names:
['Id', 'Age', 'Gender', 'Hormonal Changes', 'Family History', 'Race/Ethnicity', 'Body Weight', 'Calcium Intake', 'Vitamin D Intake', 'Physical Activity', 'Smoking', 'Alcohol Consumption', 'Medical Conditions', 'Medications', 'Prior Fractures', 'Osteoporosis']

Data types:


,dtype
Id,int64
Age,int64
Gender,object
Hormonal Changes,object
Family History,object
Race/Ethnicity,object
Body Weight,object
Calcium Intake,object
Vitamin D Intake,object
Physical Activity,object



First 5 rows:


,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
0,104866,69,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,Yes,Moderate,Rheumatoid Arthritis,Corticosteroids,Yes,1
1,101999,32,Female,Normal,Yes,Asian,Underweight,Low,Sufficient,Sedentary,No,NaN,NaN,NaN,Yes,1
2,106567,89,Female,Postmenopausal,No,Caucasian,Normal,Adequate,Sufficient,Active,No,Moderate,Hyperthyroidism,Corticosteroids,No,1
3,102316,78,Female,Normal,No,Caucasian,Underweight,Adequate,Insufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,Corticosteroids,No,1
4,101944,38,Male,Postmenopausal,Yes,African American,Normal,Low,Sufficient,Active,Yes,NaN,Rheumatoid Arthritis,NaN,Yes,1



Last 5 rows:


,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
1953,106130,19,Female,Normal,Yes,African American,Normal,Adequate,Sufficient,Sedentary,Yes,Moderate,Rheumatoid Arthritis,NaN,Yes,0
1954,106463,23,Female,Postmenopausal,Yes,Caucasian,Underweight,Low,Insufficient,Active,No,NaN,NaN,Corticosteroids,No,0
1955,103142,34,Female,Postmenopausal,No,African American,Underweight,Low,Sufficient,Sedentary,No,NaN,Hyperthyroidism,NaN,No,0
1956,105187,25,Male,Postmenopausal,No,African American,Normal,Low,Insufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,Corticosteroids,Yes,0
1957,108561,26,Female,Postmenopausal,No,African American,Underweight,Adequate,Sufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,Corticosteroids,No,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1958 entries, 0 to 1957
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Id                   1958 non-null   int64 
 1   Age                  1958 non-null   int64 
 2   Gender               1958 non-null   object
 3   Hormonal Changes     1958 non-null   object
 4   Family History       1958 non-null   object
 5   Race/Ethnicity       1958 non-null   object
 6   Body Weight          1958 non-null   object
 7   Calcium Intake       1958 non-null   object
 8   Vitamin D Intake     1958 non-null   object
 9   Physical Activity    1958 non-null   object
 10  Smoking              1958 non-null   object
 11  Alcohol Consumption  970 non-null    object
 12  Medical Conditions   1311 non-null   object
 13  Medications          973 non-null    object
 14  Prior Fractures      1958 non-null   object
 15  Osteoporosis         1958 non-null   int64 
dtypes: int

In [6]:
df.describe()

,Id,Age,Osteoporosis
count,1958.000000,1958.000000,1958.000000
mean,105515.320735,39.101124,0.500000
std,2589.407806,21.355424,0.500128
min,101008.000000,18.000000,0.000000
25%,103348.500000,21.000000,0.000000
50%,105469.000000,32.000000,0.500000
75%,107755.000000,53.000000,1.000000
max,109996.000000,90.000000,1.000000


In [7]:
df.describe(include="all")

,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
count,1958.000000,1958.000000,1958,1958,1958,1958,1958,1958,1958,1958,1958,970,1311,973,1958,1958.000000
unique,NaN,NaN,2,2,2,3,2,2,2,2,2,1,2,1,2,NaN
top,NaN,NaN,Male,Normal,No,African American,Normal,Low,Sufficient,Active,Yes,Moderate,Hyperthyroidism,Corticosteroids,Yes,NaN
freq,NaN,NaN,992,981,998,681,1027,1004,1011,1021,982,970,678,973,983,NaN
mean,105515.320735,39.101124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000
std,2589.407806,21.355424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500128
min,101008.000000,18.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
25%,103348.500000,21.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
50%,105469.000000,32.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000
75%,107755.000000,53.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [8]:
for column in df.columns:
    print(f"\n{column}")
    print(df[column].unique())


Id
[104866 101999 106567 ... 106130 105187 108561]

Age
[69 32 89 78 38 41 20 39 70 19 47 55 81 77 50 75 66 76 59 79 64 68 72 24
 90 56 35 21 31 26 88 61 25 52 53 67 23 71 80 51 65 57 43 58 46 62 82 18
 28 22 45 29 40 54 44 60 30 49 83 74 87 73 36 85 34 86 33 37 84 48 42 63
 27]

Gender
['Female' 'Male']

Hormonal Changes
['Normal' 'Postmenopausal']

Family History
['Yes' 'No']

Race/Ethnicity
['Asian' 'Caucasian' 'African American']

Body Weight
['Underweight' 'Normal']

Calcium Intake
['Low' 'Adequate']

Vitamin D Intake
['Sufficient' 'Insufficient']

Physical Activity
['Sedentary' 'Active']

Smoking
['Yes' 'No']

Alcohol Consumption
['Moderate' nan]

Medical Conditions
['Rheumatoid Arthritis' nan 'Hyperthyroidism']

Medications
['Corticosteroids' nan]

Prior Fractures
['Yes' 'No']

Osteoporosis
[1 0]


In [9]:
df.isnull().sum()

Id                       0
Age                      0
Gender                   0
Hormonal Changes         0
Family History           0
Race/Ethnicity           0
Body Weight              0
Calcium Intake           0
Vitamin D Intake         0
Physical Activity        0
Smoking                  0
Alcohol Consumption    988
Medical Conditions     647
Medications            985
Prior Fractures          0
Osteoporosis             0
dtype: int64

In [10]:
missing_percentage = (
    df.isnull().mean() * 100
).round(2)

missing_percentage

Id                      0.00
Age                     0.00
Gender                  0.00
Hormonal Changes        0.00
Family History          0.00
Race/Ethnicity          0.00
Body Weight             0.00
Calcium Intake          0.00
Vitamin D Intake        0.00
Physical Activity       0.00
Smoking                 0.00
Alcohol Consumption    50.46
Medical Conditions     33.04
Medications            50.31
Prior Fractures         0.00
Osteoporosis            0.00
dtype: float64

In [11]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
Id,0,0.00
Age,0,0.00
Gender,0,0.00
Hormonal Changes,0,0.00
Family History,0,0.00
Race/Ethnicity,0,0.00
Body Weight,0,0.00
Calcium Intake,0,0.00
Vitamin D Intake,0,0.00
Physical Activity,0,0.00


In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df[df.duplicated(keep=False)]

,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis


In [15]:
# check for duplicate IDs
df["Id"].duplicated().sum()

np.int64(209)

In [17]:
# check for duplicate IDs and display the rows with duplicate IDs
duplicate_ids = df[
    df["Id"].duplicated(keep=False)
].sort_values("Id")

duplicate_ids.head(20)

,Id,Age,Gender,Hormonal Changes,Family History,Race/Ethnicity,Body Weight,Calcium Intake,Vitamin D Intake,Physical Activity,Smoking,Alcohol Consumption,Medical Conditions,Medications,Prior Fractures,Osteoporosis
77,101027,82,Female,Normal,No,Asian,Underweight,Low,Insufficient,Active,No,Moderate,NaN,Corticosteroids,No,1
1497,101027,29,Female,Postmenopausal,Yes,Caucasian,Normal,Low,Insufficient,Active,No,Moderate,NaN,Corticosteroids,No,0
1544,101148,25,Female,Postmenopausal,No,African American,Underweight,Low,Insufficient,Active,No,Moderate,Hyperthyroidism,NaN,No,0
736,101148,21,Male,Normal,Yes,Asian,Normal,Adequate,Insufficient,Sedentary,Yes,Moderate,Rheumatoid Arthritis,Corticosteroids,No,1
720,101162,59,Female,Postmenopausal,Yes,African American,Underweight,Low,Sufficient,Active,Yes,Moderate,Hyperthyroidism,NaN,Yes,1
650,101162,25,Male,Normal,No,Caucasian,Underweight,Adequate,Sufficient,Active,Yes,NaN,Hyperthyroidism,Corticosteroids,Yes,1
647,101170,40,Male,Postmenopausal,No,Asian,Normal,Adequate,Sufficient,Active,No,NaN,Hyperthyroidism,Corticosteroids,No,1
1297,101170,35,Male,Normal,No,Caucasian,Normal,Adequate,Insufficient,Active,Yes,NaN,NaN,NaN,Yes,0
1385,101180,20,Male,Postmenopausal,Yes,Asian,Normal,Adequate,Sufficient,Sedentary,Yes,NaN,Rheumatoid Arthritis,NaN,Yes,0
1941,101180,20,Male,Postmenopausal,No,African American,Normal,Adequate,Sufficient,Active,Yes,Moderate,NaN,NaN,Yes,0


In [22]:
# check the distribution of the target variable
df["Osteoporosis"].value_counts()


Osteoporosis
1    979
0    979
Name: count, dtype: int64

In [23]:
# check the distribution of the target variable in percentage
df["Osteoporosis"].value_counts(normalize=True) * 100

Osteoporosis
1    50.0
0    50.0
Name: proportion, dtype: float64

In [25]:
# check for inconsistencie harmonal changes 
pd.crosstab(
    df["Gender"],
    df["Hormonal Changes"]
)

Hormonal Changes,Normal,Postmenopausal
Gender,,
Female,475,491
Male,506,486


In [26]:
categorical_columns = df.select_dtypes(
    include="object"
).columns

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


Gender
Gender
Male      992
Female    966
Name: count, dtype: int64

Hormonal Changes
Hormonal Changes
Normal            981
Postmenopausal    977
Name: count, dtype: int64

Family History
Family History
No     998
Yes    960
Name: count, dtype: int64

Race/Ethnicity
Race/Ethnicity
African American    681
Caucasian           646
Asian               631
Name: count, dtype: int64

Body Weight
Body Weight
Normal         1027
Underweight     931
Name: count, dtype: int64

Calcium Intake
Calcium Intake
Low         1004
Adequate     954
Name: count, dtype: int64

Vitamin D Intake
Vitamin D Intake
Sufficient      1011
Insufficient     947
Name: count, dtype: int64

Physical Activity
Physical Activity
Active       1021
Sedentary     937
Name: count, dtype: int64

Smoking
Smoking
Yes    982
No     976
Name: count, dtype: int64

Alcohol Consumption
Alcohol Consumption
NaN         988
Moderate    970
Name: count, dtype: int64

Medical Conditions
Medical Conditions
Hyperthyroidism         678
NaN

In [27]:
for column in ["Alcohol Consumption", "Medical Conditions", "Medications"]:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


Alcohol Consumption:
Alcohol Consumption
NaN         988
Moderate    970
Name: count, dtype: int64

Medical Conditions:
Medical Conditions
Hyperthyroidism         678
NaN                     647
Rheumatoid Arthritis    633
Name: count, dtype: int64

Medications:
Medications
NaN                985
Corticosteroids    973
Name: count, dtype: int64


## Exploratory Data Analysis (EDA)